# The DFT Matrix as a Unitary Change of Basis

The **discrete Fourier transform** (DFT) is usually introduced as a formula for "the frequencies in a signal". In linear-algebra terms it is something cleaner: multiplication by the $n\times n$ matrix

$$
F_{jk} = \frac{\omega^{jk}}{\sqrt n}, \qquad \omega = e^{-2\pi i/n},
$$

whose entries are the $n$-th roots of unity. This notebook shows that $F$ is **unitary**, that its columns are **sampled complex exponentials**, and that computing $Fx$ is nothing but writing $x$ in a new orthonormal basis of $\mathbb{C}^n$. Parseval's theorem then comes for free, numpy's `np.fft.fft` turns out to be $\sqrt n\,F$, and the curious fact $F^4 = I$ pins every eigenvalue of $F$ to the set $\{1, -1, i, -i\}$.

This is notebook 05 of module 04 (**Complex linear algebra**) in the **Learning Complex Numbers** series.

## 1. Introduction

A digital signal is a vector $x = (x_0, \dots, x_{n-1})$ of samples. Two bases of $\mathbb{C}^n$ are natural for it:

- the **standard basis** $e_0, \dots, e_{n-1}$ — "the value at time $j$";
- the **Fourier basis** $f_0, \dots, f_{n-1}$ — "a pure oscillation with $k$ cycles over the window".

The DFT converts coordinates from the first to the second. Because both bases are orthonormal, the conversion matrix is unitary (notebook 04): it preserves lengths, so the energy of a signal is the same whether you add it up over time or over frequency. Everything that makes the Fourier transform useful — audio spectra, JPEG compression, fast polynomial multiplication, solving PDEs — builds on this one matrix, and the FFT is just a fast algorithm for multiplying by it.

**Prerequisites**

- The $n$-th roots of unity and their sums — [Roots of unity and n-th roots](../01_polar_form_and_euler/05_roots_of_unity_and_nth_roots.ipynb).
- Hermitian inner product, orthonormal bases and coordinates $\langle e_j, v\rangle$ — [Complex vectors and inner products](03_complex_vectors_and_inner_products.ipynb).
- Unitary matrices and their eigenvalues — [Hermitian and unitary matrices](04_hermitian_and_unitary_matrices.ipynb).
- Orthogonality of sines and cosines, Fourier analysis over $\mathbb{R}$ — [Orthogonality and Fourier analysis](../../../LearningLinearAlgebra/hyperskill/11_advanced_concepts/orthogonality_and_fourier_analysis.ipynb) (LearningLinearAlgebra).

After completing this notebook you will be able to:

- Write down the DFT matrix for small $n$ and prove that it is unitary using a geometric sum of roots of unity.
- Interpret its columns as sampled complex exponentials and the DFT as a change of basis (analysis $X = Fx$, synthesis $x = F^HX$).
- Convert between the unitary $F$ and numpy's `np.fft.fft` / `ifft` normalisations, and verify Parseval's identity.
- Show $F^2$ is a reversal permutation, $F^4 = I$, and hence that the eigenvalues of $F$ lie in $\{\pm 1, \pm i\}$.

## 2. Theory

### 2.1 Definition

Index rows and columns from $0$ to $n - 1$ and let $\omega = e^{-2\pi i/n}$, a primitive $n$-th root of unity. The (unitary) **DFT matrix** is

$$
F = \frac{1}{\sqrt n}\begin{pmatrix}
1 & 1 & 1 & \cdots & 1 \\
1 & \omega & \omega^2 & \cdots & \omega^{n-1} \\
1 & \omega^2 & \omega^4 & \cdots & \omega^{2(n-1)} \\
\vdots & & & \ddots & \vdots \\
1 & \omega^{n-1} & \omega^{2(n-1)} & \cdots & \omega^{(n-1)^2}
\end{pmatrix}, \qquad F_{jk} = \frac{\omega^{jk}}{\sqrt n}.
$$

It is symmetric ($F_{jk} = F_{kj}$), and since $\omega^n = 1$ only $jk \bmod n$ matters. For $n = 4$, $\omega = e^{-i\pi/2} = -i$:

$$
F_4 = \frac12\begin{pmatrix} 1 & 1 & 1 & 1 \\ 1 & -i & -1 & i \\ 1 & -1 & 1 & -1 \\ 1 & i & -1 & -i \end{pmatrix}.
$$

(The sign in $\omega$ is a convention. We use $e^{-2\pi i/n}$ because that is what numpy and most signal-processing texts use for the forward transform.)

### 2.2 Columns are sampled complex exponentials

Column $k$ of $\sqrt n\,F$ is $(\omega^{0\cdot k}, \omega^{1\cdot k}, \dots) = \big(e^{-2\pi i\,k j/n}\big)_{j}$: the function $t \mapsto e^{-2\pi i k t}$ sampled at $t = j/n$. Its real part is $\cos(2\pi k j/n)$, its imaginary part $-\sin(2\pi kj/n)$: an oscillation with **$k$ cycles** across the window. Geometrically, as $j$ increases the entry steps around the unit circle by the angle $-2\pi k/n$ per sample.

Two features of sampling:

- **Aliasing:** frequency $k$ and $k + n$ give identical samples, so only $k = 0, \dots, n - 1$ appear.
- **Negative frequencies:** column $n - k$ is the conjugate of column $k$, i.e. frequency $n - k$ *is* frequency $-k$. A real cosine needs both $k$ and $n - k$.

### 2.3 $F$ is unitary

The $(k, l)$ entry of $F^HF$ is the inner product of columns $k$ and $l$:

$$
(F^HF)_{kl} = \frac1n\sum_{j=0}^{n-1}\overline{\omega^{jk}}\,\omega^{jl} = \frac1n\sum_{j=0}^{n-1}\omega^{j(l - k)},
$$

using $\bar\omega = \omega^{-1}$ (as $|\omega| = 1$). Let $q = \omega^{l-k}$. If $l = k$, then $q = 1$ and the sum is $n$. If $l \ne k$ (with $0 \le k, l < n$), then $q \ne 1$ but $q^n = 1$, and the geometric sum gives

$$
\sum_{j=0}^{n-1} q^j = \frac{1 - q^n}{1 - q} = 0.
$$

(Picture: the powers of $q$ are equally spaced points on the unit circle, and they balance around the origin.) Hence $F^HF = I$:

$$
\boxed{\;F^{-1} = F^H = \bar F\;}
$$

### 2.4 The DFT is a change of basis

Let $f_k$ be the $k$-th column of $F^H$ (the conjugate of column $k$ of $F$):

$$
f_k = \frac{1}{\sqrt n}\big(1, e^{2\pi i k/n}, e^{2\pi i\,2k/n}, \dots, e^{2\pi i (n-1)k/n}\big)^T.
$$

Because $F^H$ is unitary, $f_0, \dots, f_{n-1}$ is an **orthonormal basis** of $\mathbb{C}^n$, the Fourier basis. By notebook 03, the coordinates of any $x$ in this basis are

$$
X_k = \langle f_k, x\rangle = \sum_{j}\overline{(f_k)_j}\,x_j = \frac{1}{\sqrt n}\sum_{j=0}^{n-1}e^{-2\pi i jk/n}x_j = (Fx)_k.
$$

So:

$$
\text{analysis: } X = Fx, \qquad \text{synthesis: } x = F^HX = \sum_{k=0}^{n-1}X_k\,f_k.
$$

$X_k$ says how much of the oscillation with $k$ cycles is in $x$, and $|X_k|$ and $\arg X_k$ are its **amplitude** and **phase**.

### 2.5 numpy's normalisation

`np.fft.fft` omits the $1/\sqrt n$:

$$
\texttt{fft}(x)_k = \sum_{j=0}^{n-1}x_je^{-2\pi i jk/n} = \sqrt n\,(Fx)_k, \qquad
\texttt{ifft}(y)_j = \frac1n\sum_{k=0}^{n-1}y_ke^{2\pi i jk/n} = \frac{1}{\sqrt n}(F^Hy)_j.
$$

The pair is still mutual inverses ($\sqrt n \cdot \frac{1}{\sqrt n} = 1$), but `fft` alone is not unitary. Passing `norm="ortho"` gives exactly $F$ and $F^H$.

### 2.6 Parseval and Plancherel

A unitary matrix preserves inner products and norms (notebook 04), so

$$
\langle Fx, Fy\rangle = \langle x, y\rangle, \qquad \sum_{k}|X_k|^2 = \sum_j|x_j|^2.
$$

"Energy in time = energy in frequency." In numpy's normalisation: $\sum_j|x_j|^2 = \frac1n\sum_k|\texttt{fft}(x)_k|^2$.

### 2.7 $F^2$ reverses, $F^4 = I$, and the eigenvalues

Compute $(F^2)_{jl} = \frac1n\sum_k\omega^{jk}\omega^{kl} = \frac1n\sum_k\omega^{k(j + l)}$. By the same geometric sum this is $1$ when $j + l \equiv 0 \pmod n$ and $0$ otherwise. So $F^2$ is the **index-reversal permutation**

$$
(F^2x)_j = x_{-j \bmod n}: \qquad F^2(x_0, x_1, \dots, x_{n-1}) = (x_0, x_{n-1}, \dots, x_1).
$$

Reversing twice changes nothing, so $F^4 = I$. If $Fv = \lambda v$ then $v = F^4v = \lambda^4v$, hence $\lambda^4 = 1$:

$$
\lambda \in \{1, -i, -1, i\}.
$$

These are the only four values, with multiplicities roughly $n/4$ each (exact counts depend on $n \bmod 4$). Since $F$ is unitary, it is normal and has an orthonormal eigenbasis; with only four eigenvalues the eigenspaces are large and the eigenvectors are far from unique.

### 2.8 Why this basis? Shifts

Let $S$ be the cyclic shift $(Sx)_j = x_{j-1}$ (indices mod $n$), i.e. "delay by one sample". Each Fourier vector is an eigenvector of $S$:

$$
(Sf_k)_j = (f_k)_{j-1} = e^{-2\pi i k/n}(f_k)_j \quad\Rightarrow\quad Sf_k = e^{-2\pi i k/n}f_k.
$$

So $FSF^H$ is diagonal: in the Fourier basis, delaying a signal just multiplies each coefficient by a phase. Every circulant matrix (a polynomial in $S$, i.e. a cyclic convolution) is diagonalised by $F$ too — the **convolution theorem**, which module 07 uses.

## 3. Worked Examples

### Example 1 — $n = 2$

$\omega = e^{-i\pi} = -1$, so $F_2 = \frac{1}{\sqrt2}\begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix}$ — the Hadamard matrix. It is real, symmetric and unitary, with $F_2^2 = I$ already, eigenvalues $\pm 1$. $X_0 = (x_0 + x_1)/\sqrt2$ is the "average" and $X_1 = (x_0 - x_1)/\sqrt2$ the "difference".

### Example 2 — The DFT of $x = (1, 2, 3, 4)$

Using $F_4$:

$$
X = \frac12\begin{pmatrix} 1 + 2 + 3 + 4 \\ 1 - 2i - 3 + 4i \\ 1 - 2 + 3 - 4 \\ 1 + 2i - 3 - 4i \end{pmatrix} = \begin{pmatrix} 5 \\ -1 + i \\ -1 \\ -1 - i \end{pmatrix}.
$$

numpy's `fft` gives $2X = (10, -2 + 2i, -2, -2 - 2i)$. Note $X_3 = \overline{X_1}$ because $x$ is real.

Parseval: $\lVert x\rVert^2 = 1 + 4 + 9 + 16 = 30$ and $\lVert X\rVert^2 = 25 + 2 + 1 + 2 = 30$. ✓ (With numpy: $\frac14(100 + 8 + 4 + 8) = 30$.)

### Example 3 — A cosine splits into $\pm$ frequency

$x = (\cos(2\pi j/4))_{j=0}^{3} = (1, 0, -1, 0)$. Then

$$
Fx = \frac12\big(1 - 1,\ 1 + 1,\ 1 - 1,\ 1 + 1\big) = (0, 1, 0, 1).
$$

Only frequencies $1$ and $3 \equiv -1$ appear: $x = f_1 + f_3$, which is $\cos\theta = \frac12(e^{i\theta} + e^{-i\theta})$ in vector form. Check: $f_1 + f_3 = \frac12\big[(1, i, -1, -i) + (1, -i, -1, i)\big] = (1, 0, -1, 0)$. ✓

### Example 4 — $F^2$ for $n = 4$

$F_4^2$ has a $1$ wherever $j + l \equiv 0 \pmod 4$: positions $(0,0), (1,3), (2,2), (3,1)$. So $F_4^2(x_0, x_1, x_2, x_3) = (x_0, x_3, x_2, x_1)$, and applying it twice returns $x$: $F_4^4 = I$.

### Example 5 — An eigenvector with eigenvalue $1$

For any $x$, $v = x + Fx + F^2x + F^3x$ satisfies $Fv = Fx + F^2x + F^3x + F^4x = v$. Start from $x = e_0$ with $n = 4$: $Fe_0 = \frac12(1, 1, 1, 1)$, $F^2e_0 = e_0$ (reversal fixes index $0$), $F^3e_0 = F^2(Fe_0) = \frac12(1, 1, 1, 1)$. So $v = (3, 1, 1, 1)$. Check: $Fv = \frac12\big(6,\ 3 - i - 1 + i,\ 3 - 1 + 1 - 1,\ 3 + i - 1 - i\big) = (3, 1, 1, 1)$. ✓

## 4. Python Examples

### 4.1 Building $F$ and checking that it is unitary

In [ ]:
import numpy as np


def dft_matrix(n):
    indices = np.arange(n)
    omega = np.exp(-2j * np.pi / n)
    return omega ** np.outer(indices, indices) / np.sqrt(n)


F4 = dft_matrix(4)
print("2 F_4 =\n", np.round(2 * F4, 12) + (0 + 0j))       # + 0 tidies -0 signs in the display
for n in (2, 4, 7, 64):
    F = dft_matrix(n)
    print(f"n = {n:3d}: symmetric {np.allclose(F, F.T)}, unitary {np.allclose(F.conj().T @ F, np.eye(n))}, "
          f"F^-1 = conj(F) {np.allclose(np.linalg.inv(F), F.conj())}")

### 4.2 Exact check with sympy for $n = 4$ and the geometric sum

In [ ]:
import sympy as sp

n = 4
omega = sp.exp(-2 * sp.pi * sp.I / n)
F = sp.Matrix(n, n, lambda j, k: omega ** (j * k)) / sp.sqrt(n)
print("2F =", sp.simplify(2 * F))
print("F^H F =", sp.simplify(F.H * F))

for shift in range(n):
    total = sum(sp.exp(-2 * sp.pi * sp.I * shift * j / n) for j in range(n))
    print(f"sum_j omega^(j*{shift}) =", sp.simplify(total))

### 4.3 Matching `np.fft.fft`

`fft` is $\sqrt n\,F$, `ifft` is $F^H/\sqrt n$, and `norm="ortho"` gives $F$ and $F^H$ exactly.

In [ ]:
import numpy as np


def dft_matrix(n):
    indices = np.arange(n)
    return np.exp(-2j * np.pi / n) ** np.outer(indices, indices) / np.sqrt(n)


x = np.array([1.0, 2.0, 3.0, 4.0])
F = dft_matrix(4)
print("F x              =", np.round(F @ x, 12) + 0)
print("np.fft.fft(x)    =", np.fft.fft(x))
print("fft = sqrt(n) F x:", np.allclose(np.fft.fft(x), np.sqrt(4) * F @ x))
print("ortho fft = F x  :", np.allclose(np.fft.fft(x, norm="ortho"), F @ x))

rng = np.random.default_rng(0)
y = rng.normal(size=16) + 1j * rng.normal(size=16)
F16 = dft_matrix(16)
print("ifft = F^H y / sqrt(n):", np.allclose(np.fft.ifft(y), F16.conj().T @ y / 4))
print("ifft(fft(y)) = y      :", np.allclose(np.fft.ifft(np.fft.fft(y)), y))

### 4.4 Change of basis and Parseval

Coordinates $X_k = \langle f_k, x\rangle$ computed one inner product at a time agree with $Fx$, and the synthesis $\sum X_kf_k$ rebuilds $x$.

In [ ]:
import numpy as np

n = 8
rng = np.random.default_rng(1)
x = rng.normal(size=n)

fourier_basis = [np.exp(2j * np.pi * k * np.arange(n) / n) / np.sqrt(n) for k in range(n)]
coordinates = np.array([np.vdot(f_k, x) for f_k in fourier_basis])
rebuilt = sum(X_k * f_k for X_k, f_k in zip(coordinates, fourier_basis))

print("<f_k, x> = ortho fft:", np.allclose(coordinates, np.fft.fft(x, norm="ortho")))
print("sum X_k f_k = x     :", np.allclose(rebuilt, x))
print("real x => X_{n-k} = conj(X_k):", np.allclose(coordinates[1:][::-1], np.conj(coordinates[1:])))
print(f"Parseval: ||x||^2 = {np.sum(np.abs(x)**2):.6f},  ||X||^2 = {np.sum(np.abs(coordinates)**2):.6f},  "
      f"(1/n)||fft x||^2 = {np.sum(np.abs(np.fft.fft(x))**2) / n:.6f}")

y = rng.normal(size=n) + 1j * rng.normal(size=n)
print("Plancherel <Fx, Fy> = <x, y>:", np.isclose(np.vdot(np.fft.fft(x, norm='ortho'), np.fft.fft(y, norm='ortho')), np.vdot(x, y)))

### 4.5 $F^2$, $F^4$ and the eigenvalues

In [ ]:
import numpy as np


def dft_matrix(n):
    indices = np.arange(n)
    return np.exp(-2j * np.pi / n) ** np.outer(indices, indices) / np.sqrt(n)


F = dft_matrix(6)
x = np.arange(6) * 1.0
print("F^2 x =", np.round((F @ F @ x).real, 12) + 0.0, "  for x =", x)
print("F^4 = I:", np.allclose(np.linalg.matrix_power(F, 4), np.eye(6)))

print("\nmultiplicities of the eigenvalues 1, -1, -i, i:")
for n in range(1, 13):
    eigenvalues = np.linalg.eigvals(dft_matrix(n))
    counts = [int(np.sum(np.abs(eigenvalues - target) < 1e-8)) for target in (1, -1, -1j, 1j)]
    print(f"  n = {n:2d}: {counts}   (total {sum(counts)})")

### 4.6 Fourier vectors diagonalise the cyclic shift

In [ ]:
import numpy as np


def dft_matrix(n):
    indices = np.arange(n)
    return np.exp(-2j * np.pi / n) ** np.outer(indices, indices) / np.sqrt(n)


n = 8
shift = np.roll(np.eye(n), 1, axis=0)                  # (S x)_j = x_{j-1}
print("S e_0 = e_1:", np.allclose(shift @ np.eye(n)[0], np.eye(n)[1]))

F = dft_matrix(n)
D = F @ shift @ F.conj().T
print("F S F^H is diagonal:", np.allclose(D, np.diag(np.diag(D))))
print("diagonal = e^{-2 pi i k / n}:", np.allclose(np.diag(D), np.exp(-2j * np.pi * np.arange(n) / n)))

## 5. Visualizations

### 5.1 The entries of $F$ are roots of unity

For $n = 8$, each column of $\sqrt n\,F$ walks around the unit circle through the 8-th roots of unity. Column $k$ steps by $-2\pi k/8$: column 1 visits every root once (clockwise), column 2 visits every other root twice, column 4 just alternates $\pm 1$.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

n = 8
j = np.arange(n)
circle = np.exp(1j * np.linspace(0, 2 * np.pi, 300))

fig, axes = plt.subplots(1, 4, figsize=(17, 4.8))
for ax, k in zip(axes, (1, 2, 3, 4)):
    entries = np.exp(-2j * np.pi * j * k / n)
    ax.plot(circle.real, circle.imag, color="lightgray")
    ax.plot(entries.real, entries.imag, "-", color="tab:blue", alpha=0.4)
    ax.plot(entries.real, entries.imag, "o", color="tab:blue")
    # one label per visited point, listing every j that lands there
    labels = {}
    for index, entry in enumerate(entries):
        labels.setdefault((round(entry.real, 6), round(entry.imag, 6)), []).append(str(index))
    for (re_part, im_part), indices in labels.items():
        ax.text(1.25 * re_part, 1.25 * im_part, ",".join(indices), ha="center", va="center", fontsize=9)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlim(-1.6, 1.6)
    ax.set_ylim(-1.6, 1.6)
    ax.set_aspect("equal")
    ax.set_title(f"column k = {k}: $\\omega^{{jk}}$, j = 0..7")
    ax.set_xlabel("Re")
    ax.set_ylabel("Im")
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 5.2 The matrix as a picture

Real and imaginary parts of $\sqrt n\,F$ for $n = 32$. Each row and column is a cosine (left) or minus-sine (right) wave; the frequency increases towards the middle and then falls again (frequencies $k$ and $n - k$ look alike: aliasing).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

n = 32
indices = np.arange(n)
scaled_F = np.exp(-2j * np.pi * np.outer(indices, indices) / n)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, part, title in [(axes[0], scaled_F.real, "$\\mathrm{Re}(\\sqrt{n}\\,F) = \\cos(2\\pi jk/n)$"),
                        (axes[1], scaled_F.imag, "$\\mathrm{Im}(\\sqrt{n}\\,F) = -\\sin(2\\pi jk/n)$")]:
    image = ax.imshow(part, cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_title(title)
    ax.set_xlabel("column k (frequency)")
    ax.set_ylabel("row j (time)")
    fig.colorbar(image, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

### 5.3 Columns are sampled waves

The real and imaginary parts of the Fourier basis vectors $f_k$ for $n = 8$, drawn as samples (blue stems) of the continuous waves $\cos(2\pi kt)$ and $\sin(2\pi kt)$ (gray) on $t \in [0, 1)$. Note $f_7$ (last column): its samples equal those of the slow frequency $-1$ wave (orange, dashed), although the continuous wave with 7 cycles is much faster — aliasing.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

n = 8
t_fine = np.linspace(0, 1, 400)
t_samples = np.arange(n) / n

fig, axes = plt.subplots(2, 4, figsize=(17, 6.5), sharex=True, sharey=True)
for column, k in enumerate((0, 1, 2, 7)):
    samples = np.exp(2j * np.pi * k * t_samples)
    for row, (part, function, name) in enumerate([(samples.real, np.cos, "Re"), (samples.imag, np.sin, "Im")]):
        ax = axes[row, column]
        ax.plot(t_fine, function(2 * np.pi * k * t_fine), color="lightgray", label=f"continuous, {k} cycles")
        if k == 7:
            ax.plot(t_fine, function(-2 * np.pi * t_fine), "--", color="tab:orange", label="frequency $-1$")
        ax.stem(t_samples, part, basefmt=" ")
        ax.set_title(f"{name} $f_{{{k}}}$ (times $\\sqrt{{n}}$)")
        ax.grid(True, alpha=0.3)
for ax in axes[1]:
    ax.set_xlabel("t = j / n")
plt.tight_layout()
plt.show()

### 5.4 A signal and its coordinates in the Fourier basis

A 64-sample signal made of a frequency-5 cosine (amplitude 2) and a frequency-12 sine (amplitude 1), plus a constant offset 0.5. The unitary DFT concentrates it in five coefficients: $k = 0$, $\pm 5$ and $\pm 12$ (the negative frequencies appear at $64 - 5 = 59$ and $64 - 12 = 52$). The two energy totals agree — Parseval.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

n = 64
j = np.arange(n)
x = 0.5 + 2 * np.cos(2 * np.pi * 5 * j / n) + np.sin(2 * np.pi * 12 * j / n)
X = np.fft.fft(x, norm="ortho")

fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
axes[0].stem(j, x, basefmt=" ")
axes[0].set_title(f"signal $x_j$  (energy $\\|x\\|^2$ = {np.sum(np.abs(x)**2):.2f})")
axes[0].set_xlabel("j")
axes[0].set_ylabel("$x_j$")
axes[0].grid(True, alpha=0.3)

axes[1].stem(np.arange(n), np.abs(X), basefmt=" ")
axes[1].set_title(f"$|X_k| = |(Fx)_k|$  (energy $\\|X\\|^2$ = {np.sum(np.abs(X)**2):.2f})")
axes[1].set_xlabel("k")
axes[1].set_ylabel("$|X_k|$")
axes[1].grid(True, alpha=0.3)
for k in (0, 5, 12, 52, 59):
    axes[1].annotate(f"{k}", (k, abs(X[k])), textcoords="offset points", xytext=(0, 6), ha="center", fontsize=9)
plt.tight_layout()
plt.show()

large = np.flatnonzero(np.abs(X) > 1e-9)
print("nonzero coefficients at k =", large)
print("X at those k:", np.round(X[large], 4) + (0 + 0j))

### 5.5 The eigenvalues of $F$

Eigenvalues of $F$ for $n = 16$ and $n = 17$, drawn on the unit circle: all of them sit on the four points $\pm 1, \pm i$ (jittered radially so the multiplicities are visible).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def dft_matrix(n):
    indices = np.arange(n)
    return np.exp(-2j * np.pi / n) ** np.outer(indices, indices) / np.sqrt(n)


circle = np.exp(1j * np.linspace(0, 2 * np.pi, 300))
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for ax, n in zip(axes, (16, 17)):
    eigenvalues = np.linalg.eigvals(dft_matrix(n))
    ax.plot(circle.real, circle.imag, color="lightgray")
    for target in (1, -1, 1j, -1j):
        matches = eigenvalues[np.abs(eigenvalues - target) < 1e-8]
        radii = 1 + 0.07 * np.arange(len(matches))
        ax.plot(radii * np.real(target), radii * np.imag(target), "o", color="tab:blue")
        ax.annotate(f"{len(matches)}x", (1.55 * np.real(target), 1.55 * np.imag(target)), ha="center", va="center", fontsize=12)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlim(-1.9, 1.9)
    ax.set_ylim(-1.9, 1.9)
    ax.set_aspect("equal")
    ax.set_title(f"eigenvalues of $F$, n = {n}")
    ax.set_xlabel("Re")
    ax.set_ylabel("Im")
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Exercises

### Basic

1. Write down $F$ for $n = 3$ (with $\omega = e^{-2\pi i/3} = -\frac12 - \frac{\sqrt3}{2}i$) and verify that its first two columns are orthogonal.
2. Compute $Fx$ for $n = 4$ and $x = (1, 1, 1, 1)$, and for $x = e_0 = (1, 0, 0, 0)$. Interpret each answer as a statement about the Fourier basis.
3. Compute `np.fft.fft(x)` by hand for $x = (0, 1, 0, -1)$, convert it to $Fx$, and verify Parseval.

### Intermediate

4. **Draw it first, then compute.** For $n = 8$, sketch on the unit circle the entries $\omega^{jk}$, $j = 0, \dots, 7$, of columns $k = 1$, $k = 2$ and $k = 4$ of $\sqrt8\,F$. Which column visits only two points? Which visits each of four points twice? Then compute the columns and check.
5. Show that column $n - k$ of $F$ is the complex conjugate of column $k$. Deduce that for real $x$, $X_{n-k} = \overline{X_k}$.
6. Write out $F_4^2$ and $F_4^3$. Use $F^4 = I$ to express $F^{-1}$ as a power of $F$, and compare with $F^{-1} = \bar F$.

### Advanced

7. Prove for general $n$ that $F$ is unitary, explaining exactly where $q \ne 1$ and $q^n = 1$ are used.
8. Show that $v_\lambda = \sum_{m=0}^{3}\lambda^{-m}F^mx$ satisfies $Fv_\lambda = \lambda v_\lambda$ for each $\lambda \in \{1, -1, i, -i\}$. For $n = 4$ and $x = e_0$, compute $v_{-1}$.
9. Let $S$ be the cyclic shift $(Sx)_j = x_{j-1}$. Prove $FSF^H = \operatorname{diag}(1, \omega, \omega^2, \dots, \omega^{n-1})$ and interpret: what does delaying a signal by one sample do to its Fourier coefficients?

## 7. Solutions

1. $F_3 = \frac{1}{\sqrt3}\begin{pmatrix} 1 & 1 & 1 \\ 1 & \omega & \omega^2 \\ 1 & \omega^2 & \omega \end{pmatrix}$ (as $\omega^4 = \omega$). Columns 0 and 1: $\frac13(1 + \omega + \omega^2) = 0$, because the three cube roots of unity sum to zero.
2. $F(1, 1, 1, 1) = (2, 0, 0, 0)$: the constant vector is $2f_0$, pure frequency $0$. $Fe_0 = \frac12(1, 1, 1, 1)$: a single spike contains every frequency with equal weight.
3. $\texttt{fft}(x)_k = \sum_j x_j(-i)^{jk} = (-i)^k - (-i)^{3k}$: $k = 0$: $0$; $k = 1$: $-i - i = -2i$; $k = 2$: $-1 - (-1) = 0$; $k = 3$: $i - (-i) = 2i$. So $\texttt{fft}(x) = (0, -2i, 0, 2i)$ and $Fx = (0, -i, 0, i)$. Parseval: $\lVert x\rVert^2 = 2 = 1 + 1$. ✓ (This is the sine $\sin(2\pi j/4) = \frac{1}{2i}(e^{i\theta} - e^{-i\theta})$, i.e. $x = -if_1 + if_3$.)
4. Column $k$ steps by $-2\pi k/8$. $k = 1$: all eight roots, clockwise, once each. $k = 2$: steps of $-90°$, visiting $1, -i, -1, i$ twice each. $k = 4$: steps of $180°$, alternating $1, -1$ — only two points.
5. $\omega^{j(n-k)} = \omega^{jn}\omega^{-jk} = \omega^{-jk} = \overline{\omega^{jk}}$. Then $X_{n-k} = \frac{1}{\sqrt n}\sum_j\omega^{j(n-k)}x_j = \frac{1}{\sqrt n}\sum_j\overline{\omega^{jk}}x_j = \overline{X_k}$ when every $x_j$ is real.
6. $F_4^2 = \begin{pmatrix} 1 & 0 & 0 & 0 \\ 0 & 0 & 0 & 1 \\ 0 & 0 & 1 & 0 \\ 0 & 1 & 0 & 0 \end{pmatrix}$ and $F_4^3 = F_4^2F_4$ is $F_4$ with rows $1$ and $3$ swapped: $\frac12\begin{pmatrix} 1 & 1 & 1 & 1 \\ 1 & i & -1 & -i \\ 1 & -1 & 1 & -1 \\ 1 & -i & -1 & i \end{pmatrix} = \bar F_4$. Since $F^4 = I$, $F^{-1} = F^3$, consistent with $F^{-1} = \bar F$.
7. $(F^HF)_{kl} = \frac1n\sum_j q^j$ with $q = \omega^{l-k}$. For $k = l$, $q = 1$ and the sum is $n$. For $k \ne l$, $0 < |l - k| < n$, so $q = e^{-2\pi i(l-k)/n} \ne 1$ — this is where $q \ne 1$ is needed to divide by $1 - q$ — and $q^n = e^{-2\pi i(l-k)} = 1$ makes the numerator $1 - q^n$ vanish. So $F^HF = I$.
8. $Fv_\lambda = \sum_{m=0}^{3}\lambda^{-m}F^{m+1}x = \lambda\sum_{m=0}^{3}\lambda^{-(m+1)}F^{m+1}x = \lambda\sum_{m'=1}^{4}\lambda^{-m'}F^{m'}x = \lambda v_\lambda$, because the $m' = 4$ term $\lambda^{-4}F^4x = x$ equals the $m = 0$ term. For $\lambda = -1$, $x = e_0$: $v_{-1} = e_0 - Fe_0 + F^2e_0 - F^3e_0 = 2e_0 - (1, 1, 1, 1) = (1, -1, -1, -1)$. Check: $F(1, -1, -1, -1) = \frac12(-2,\ 1 + i + 1 - i,\ 1 + 1 - 1 + 1,\ 1 - i + 1 + i) = (-1, 1, 1, 1)$. ✓
9. From §2.8, $Sf_k = \omega^kf_k$ (as $e^{-2\pi i k/n} = \omega^k$). The columns of $F^H$ are the $f_k$, so $SF^H = F^H\operatorname{diag}(\omega^k)$, and multiplying on the left by $F = (F^H)^{-1}$ gives $FSF^H = \operatorname{diag}(\omega^k)$. Delaying by one sample multiplies coefficient $X_k$ by $\omega^k = e^{-2\pi i k/n}$: amplitudes $|X_k|$ unchanged, phases rotated by $-2\pi k/n$ — the higher the frequency, the bigger the phase shift.

Numerical checks:

In [ ]:
import numpy as np


def dft_matrix(n):
    indices = np.arange(n)
    return np.exp(-2j * np.pi / n) ** np.outer(indices, indices) / np.sqrt(n)


F3 = dft_matrix(3)
print("Ex 1: <col0, col1> =", np.round(np.vdot(F3[:, 0], F3[:, 1]), 12) + 0)
F4 = dft_matrix(4)
print("Ex 2:", np.round(F4 @ np.ones(4), 12) + 0, np.round(F4 @ np.eye(4)[0], 12) + 0)
x = np.array([0, 1, 0, -1])
print("Ex 3: fft =", np.round(np.fft.fft(x), 12) + 0, " Fx =", np.round(F4 @ x, 12) + 0,
      " Parseval:", np.isclose(np.sum(np.abs(F4 @ x) ** 2), np.sum(x**2)))
for k in (1, 2, 4):
    print(f"Ex 4: k = {k}: distinct entries = {len(np.unique(np.round(np.sqrt(8) * dft_matrix(8)[:, k], 8)))}")
F8 = dft_matrix(8)
print("Ex 5: column n-k = conj(column k):", all(np.allclose(F8[:, 8 - k], np.conj(F8[:, k])) for k in range(1, 8)))
print("Ex 6: F^2 =", np.round(np.linalg.matrix_power(F4, 2).real, 12).astype(int).tolist(),
      " F^3 = conj(F):", np.allclose(np.linalg.matrix_power(F4, 3), F4.conj()))
v = np.array([1, -1, -1, -1])
print("Ex 8: F v_{-1} = -v_{-1}:", np.allclose(F4 @ v, -v))
shift = np.roll(np.eye(8), 1, axis=0)
print("Ex 9: F S F^H = diag(omega^k):", np.allclose(F8 @ shift @ F8.conj().T, np.diag(np.exp(-2j * np.pi * np.arange(8) / 8))))

## 8. Conclusion

- The unitary DFT matrix is $F_{jk} = \omega^{jk}/\sqrt n$ with $\omega = e^{-2\pi i/n}$; its entries are $n$-th roots of unity and it is symmetric.
- $F^HF = I$ follows from one geometric sum: $n$ equally spaced points on the unit circle add to zero. So $F^{-1} = F^H = \bar F$.
- Its columns (conjugated) are the orthonormal **Fourier basis** of sampled complex exponentials $f_k$; the DFT $X = Fx$ gives the coordinates $X_k = \langle f_k, x\rangle$, and $x = F^HX = \sum X_kf_k$.
- `np.fft.fft` $= \sqrt n\,F$ and `np.fft.ifft` $= F^H/\sqrt n$; `norm="ortho"` gives the unitary pair.
- **Parseval**: $\lVert Fx\rVert = \lVert x\rVert$ (energy in time equals energy in frequency); with numpy's default, divide $\sum|\texttt{fft}|^2$ by $n$.
- $F^2$ reverses indices mod $n$, $F^4 = I$, so every eigenvalue of $F$ is one of $1, -1, i, -i$. The Fourier vectors diagonalise the cyclic shift — the root of the convolution theorem used in module 07.

This completes module 04: complex numbers as matrices, complex eigenvalues as rotations, the Hermitian inner product, Hermitian and unitary matrices, and the DFT as the unitary matrix built from roots of unity.

**Next:** module 05 begins complex *analysis* with [Limits and continuity in $\mathbb{C}$](../05_complex_differentiation/01_limits_and_continuity.ipynb) — what it means for a complex function to approach a value as $z$ approaches a point from every direction in the plane.